# PGA Tour player stats for one season

In [ ]:
%pip install requests pandas python-dotenv pymysql sqlalchemy

In [2]:
import os
import json #handle API response
import requests #send requests to API
import pandas as pd #data transformation
from mysql import connector #connect to mySQL
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
ACCESS_LEVEL = "trial"
GOLF_TOUR = "pga"
LANGUAGE_CODE = "en"
SEASON_YEAR = 2026
FORMAT = "json"

In [5]:
url = f"https://api.sportradar.com/golf/{ACCESS_LEVEL}/{GOLF_TOUR}/v3/{LANGUAGE_CODE}/{SEASON_YEAR}/players/statistics.{FORMAT}"
headers = {
    "x-api-key": API_KEY,
    "x-api-host": API_HOST
}
querystring = {
    "season": SEASON_YEAR,
    "tour": GOLF_TOUR
}

# ------- EXTRACT --------

In [6]:
response = requests.get(url=url,
                        headers=headers,
                        params=querystring)

In [7]:
payload = response.json()

In [8]:
formatted_response = json.dumps(payload, indent=4)
print(formatted_response)

{
    "tour": {
        "id": "b52068af-28e4-4e91-bdbb-037591b0ff84",
        "alias": "pga",
        "name": "PGA Tour",
        "currency": "USD"
    },
    "season": {
        "id": "e4694c64-017f-432f-b80e-28bcec314ca6",
        "year": 2026
    },
    "players": [
        {
            "id": "da226913-b804-48de-adbf-96e956eb75ac",
            "first_name": "Rory",
            "last_name": "McIlroy",
            "country": "NORTHERN IRELAND",
            "abbr_name": "R.McIlroy",
            "statistics": {
                "events_played": 14,
                "first_place": 1,
                "second_place": 1,
                "third_place": 1,
                "top_10": 6,
                "top_25": 9,
                "cuts": 0,
                "cuts_made": 14,
                "withdrawals": 1,
                "points": 2095.0,
                "points_rank": 4,
                "earnings": 11573749.0,
                "earnings_rank": 5,
                "drive_avg": 330.3,
           

# --------- TRANSFORM ------------

In [9]:
players_list = payload["players"]
players_list

[{'id': 'da226913-b804-48de-adbf-96e956eb75ac',
  'first_name': 'Rory',
  'last_name': 'McIlroy',
  'country': 'NORTHERN IRELAND',
  'abbr_name': 'R.McIlroy',
  'statistics': {'events_played': 14,
   'first_place': 1,
   'second_place': 1,
   'third_place': 1,
   'top_10': 6,
   'top_25': 9,
   'cuts': 0,
   'cuts_made': 14,
   'withdrawals': 1,
   'points': 2095.0,
   'points_rank': 4,
   'earnings': 11573749.0,
   'earnings_rank': 5,
   'drive_avg': 330.3,
   'drive_acc': 54.65,
   'gir_pct': 68.62,
   'putt_avg': 1.739,
   'sand_saves_pct': 56.04,
   'birdies_per_round': 4.31,
   'holes_per_eagle': 138.9,
   'world_rank': 2,
   'strokes_gained': 0.243,
   'hole_proximity_avg': '36\' 11"',
   'scrambling_pct': 59.34,
   'scoring_avg': 69.342,
   'strokes_gained_tee_green': 1.098,
   'strokes_gained_total': 1.341,
   'total_driving': 137,
   'all_around_ranking': 7}},
 {'id': 'da8b4888-9330-46d0-8f8a-f76fae949dba',
  'first_name': 'Jackson',
  'last_name': 'Suber',
  'country': 'UNITE

In [10]:
formatted_players_list = json.dumps(players_list, indent=4)
print(formatted_players_list)

[
    {
        "id": "da226913-b804-48de-adbf-96e956eb75ac",
        "first_name": "Rory",
        "last_name": "McIlroy",
        "country": "NORTHERN IRELAND",
        "abbr_name": "R.McIlroy",
        "statistics": {
            "events_played": 14,
            "first_place": 1,
            "second_place": 1,
            "third_place": 1,
            "top_10": 6,
            "top_25": 9,
            "cuts": 0,
            "cuts_made": 14,
            "withdrawals": 1,
            "points": 2095.0,
            "points_rank": 4,
            "earnings": 11573749.0,
            "earnings_rank": 5,
            "drive_avg": 330.3,
            "drive_acc": 54.65,
            "gir_pct": 68.62,
            "putt_avg": 1.739,
            "sand_saves_pct": 56.04,
            "birdies_per_round": 4.31,
            "holes_per_eagle": 138.9,
            "world_rank": 2,
            "strokes_gained": 0.243,
            "hole_proximity_avg": "36' 11\"",
            "scrambling_pct": 59.34,
       

In [ ]:
McIlroy_wins = players_list[0]["statistics"]["first_place"]
print(McIlroy_wins)

1


In [12]:
rows = []
column_names = ['first_name', 'last_name', 'country', 'world_rank', 'events_played', 'first_place', 'second_place', 'top_10', 'top_25', 'cuts_made', 'fed_ex_rank', 'strokes_gained_total']

for player in players_list:
    first_name              = player['first_name']
    last_name               = player['last_name']
    country                 = player['country']
    world_rank              = player['statistics']['world_rank']
    points_rank             = player['statistics']['points_rank']
    events_played           = player['statistics']['events_played']
    first_place             = player['statistics']['first_place']
    second_place            = player['statistics']['second_place']
    top_10                  = player['statistics']['top_10']
    top_25                  = player['statistics']['top_25']
    cuts_made               = player['statistics']['cuts_made']
    FedEx_rank              = player['statistics']['points_rank']
    strokes_gained_total    = player['statistics']['strokes_gained_total']

    tuple_of_player_data = (first_name, last_name, country, world_rank, events_played, first_place, second_place, top_10, top_25, cuts_made, FedEx_rank, strokes_gained_total)

    rows.append(tuple_of_player_data)



In [13]:
print(rows)

[('Rory', 'McIlroy', 'NORTHERN IRELAND', 2, 14, 1, 1, 6, 9, 14, 4, 1.341), ('Jackson', 'Suber', 'UNITED STATES', 94, 20, 0, 0, 3, 6, 13, 74, 0.475), ('Nicolai', 'Højgaard', 'DENMARK', 42, 21, 0, 2, 4, 8, 16, 35, 0.691), ('Kris', 'Ventura', 'NORWAY', 195, 19, 0, 1, 3, 5, 7, 107, 0.195), ('Gordon', 'Sargent', 'UNITED STATES', 331, 18, 0, 1, 2, 3, 7, 132, -0.694), ('Rasmus', 'Neergaard-Petersen', 'DENMARK', 72, 18, 0, 0, 2, 4, 13, 102, 0.244), ('Henrik', 'Norlander', 'SWEDEN', 561, 7, 0, 0, 0, 0, 2, 206, -1.842), ('Fabián', 'Gómez', 'ARGENTINA', 445, 7, 0, 0, 0, 0, 5, 199, -0.698), ('Keegan', 'Bradley', 'UNITED STATES', 50, 20, 0, 0, 0, 6, 16, 77, 0.459), ('Sam', 'Ryder', 'UNITED STATES', 283, 12, 0, 0, 0, 2, 8, 153, 0.561), ('Alejandro', 'Tosti', 'ARGENTINA', 493, 22, 0, 0, 0, 0, 4, 188, -1.597), ('Cameron', 'Champ', 'UNITED STATES', 441, 8, 0, 0, 0, 0, 3, 201, -1.063), ('Jackson', 'Koivun', 'UNITED STATES', 58, 8, 1, 0, 3, 5, 7, 62, 1.583), ('Kevin', 'Yu', 'CHINESE TAIPEI', 105, 21, 0, 

In [14]:
df = pd.DataFrame(rows, columns=column_names)

In [15]:
df.head()

,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total
0,Rory,McIlroy,NORTHERN IRELAND,2,14,1,1,6,9,14,4,1.341
1,Jackson,Suber,UNITED STATES,94,20,0,0,3,6,13,74,0.475
2,Nicolai,Højgaard,DENMARK,42,21,0,2,4,8,16,35,0.691
3,Kris,Ventura,NORWAY,195,19,0,1,3,5,7,107,0.195
4,Gordon,Sargent,UNITED STATES,331,18,0,1,2,3,7,132,-0.694


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   first_name            219 non-null    str    
 1   last_name             219 non-null    str    
 2   country               219 non-null    str    
 3   world_rank            219 non-null    int64  
 4   events_played         219 non-null    int64  
 5   first_place           219 non-null    int64  
 6   second_place          219 non-null    int64  
 7   top_10                219 non-null    int64  
 8   top_25                219 non-null    int64  
 9   cuts_made             219 non-null    int64  
 10  fed_ex_rank           219 non-null    int64  
 11  strokes_gained_total  219 non-null    float64
dtypes: float64(1), int64(8), str(3)
memory usage: 20.7 KB


In [17]:
df["country"] = df["country"].str.title()
df.head()

,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total
0,Rory,McIlroy,Northern Ireland,2,14,1,1,6,9,14,4,1.341
1,Jackson,Suber,United States,94,20,0,0,3,6,13,74,0.475
2,Nicolai,Højgaard,Denmark,42,21,0,2,4,8,16,35,0.691
3,Kris,Ventura,Norway,195,19,0,1,3,5,7,107,0.195
4,Gordon,Sargent,United States,331,18,0,1,2,3,7,132,-0.694


In [18]:
events = df["events_played"].where(df["events_played"] > 0)

df["win_pct"] = (df["first_place"] / events * 100).round(2)
df["top_10_pct"] = (df["top_10"] / events * 100).round(2)

df.head()


,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total,win_pct,top_10_pct
0,Rory,McIlroy,Northern Ireland,2,14,1,1,6,9,14,4,1.341,7.14,42.86
1,Jackson,Suber,United States,94,20,0,0,3,6,13,74,0.475,0.00,15.00
2,Nicolai,Højgaard,Denmark,42,21,0,2,4,8,16,35,0.691,0.00,19.05
3,Kris,Ventura,Norway,195,19,0,1,3,5,7,107,0.195,0.00,15.79
4,Gordon,Sargent,United States,331,18,0,1,2,3,7,132,-0.694,0.00,11.11


# ------------ LOAD ----------------

In [19]:
MY_SQL_HOST = os.getenv("MYSQL_HOST")
MY_SQL_PORT = os.getenv("MYSQL_PORT")
MY_SQL_USER = os.getenv("MYSQL_USER")
MY_SQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MY_SQL_DATABASE = os.getenv("MYSQL_DATABASE")

engine = create_engine(
    f"mysql+pymysql://{MY_SQL_USER}:{MY_SQL_PASSWORD}@{MY_SQL_HOST}:{MY_SQL_PORT}/{MY_SQL_DATABASE}"
)


table_name = 'players_statistics'
df.to_sql(table_name,engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table {table_name} in database {MY_SQL_DATABASE}")

Data successfully loaded into table players_statistics in database pga_db
